# Extracting embeddings from a model


## sparse_128_L32_b4_cw6

In [ ]:
from Encoding.latent_embeddings_128 import extract_embeddings

out = extract_embeddings("/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_128_L32_b4_cw6",
                        ["/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/AIIB23_128",
                         "/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/ATM22_128"],)          # one dir, or a list of dirs

enc_output = out["mu"]  # (N, num_latents) -> your clustering matrix

## sparse_256_L32_b4_cw6

In [ ]:
from Encoding.latent_embeddings_256 import extract_embeddings

out = extract_embeddings("/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_256_L32_b4_cw6",
                        ["/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/AIIB23_256^3",
                         "/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/ATM22_256^3"],)          # one dir, or a list of dirs

enc_output = out["mu"]  # (N, num_latents) -> your clustering matrix

## AE_128_L32_b2_cw6

In [ ]:
from Encoding.latent_embeddings_128_AE import extract_embeddings

out = extract_embeddings("/home/ids/gmargari-24/airway_project/Data/vae_runs/AE_128_L32_b2_cw6",
                        ["/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/AIIB23_128",
                         "/home/ids/gmargari-24/airway_project/Data/Registered_on_Template_22_23/Affine_registered/ATM22_128"],)          # one dir, or a list of dirs

enc_output = out["z"]  

In [ ]:
enctxt = np.savetxt("/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_256_L32_b4_cw6/enc0.txt", enc_output)

In [ ]:
enc_output = np.loadtxt("/home/ids/gmargari-24/airway_project/Data/vae_runs/sparse_256_L32_b4_cw6/enc0.txt")
# enc_output = enc_output.astype(int) # for autoencoder only

# Loading DGCI embeddings:

In [ ]:
import numpy as np                                          # already imported in cell 1

codes_path = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/latent_codes.npy"

enc_output = np.load(codes_path).astype(np.float64)         # (419, 128); float64 keeps ward's sums stable

# dendrogram, Fig. 4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as shc
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import pairwise_distances

plt.rcParams['lines.linewidth'] = 20
fig = plt.figure(figsize=(100, 60))
fig.patch.set_facecolor('slateblue')
plt.xlabel("Cluster Index", fontsize=250, color='white')
plt.ylabel("Euclidean Distances", fontsize=250, color='white')

# --- compute linkage once, and derive the 2-cluster line height from the data ---
Z = shc.linkage(enc_output, method='ward')
thr = (Z[-1, 2] + Z[-2, 2]) / 2          # midpoint between the last two merges = 2-cluster cut

dend = shc.dendrogram(Z,
                      truncate_mode  ='lastp',  # show only the last p merged clusters
                      p              =30,  # show only the last p merged clusters
                      leaf_rotation  =90.,
                      leaf_font_size =12.,
                      show_contracted=True,
                      # color_threshold=5500,
                      )
ax = plt.gca()
ax.tick_params(axis='x', which='major', labelsize=150, colors='white')
ax.tick_params(axis='y', which='major', labelsize=150, colors='white')

# Add a horizontal line for 2 clusters
ax.axhline(y=thr, linestyle='--', color='red', linewidth=15)

# Get the x-positions of the vertical lines
x_vals = dend['icoord']

# Iterate over the x-positions and find the intersection points with the horizontal line
for i, x in enumerate(x_vals):
    for j in range(len(x)):
        y = dend['dcoord'][i][j]
        if abs(y - thr) < 1e-5:  # check if the y-coordinate is close to the horizontal line
            ax.annotate('', xy=(x[j], y), xytext=(x[j], y + 20), arrowprops=dict(facecolor='white', edgecolor='white', arrowstyle='-|>,head_width=1,head_length=3'), fontsize=150)

plt.savefig('/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/HC.png')
plt.show()

# average intra-cluster distance, Eq. 1

In [ ]:
avg_dist = []
for c in range(1, 50):
  cluster = AgglomerativeClustering(n_clusters=c, metric='euclidean', linkage='ward')
  members = cluster.fit_predict(enc_output)
  k = np.unique(members)
  num_clusters = k.shape[0]
  sum = 0
  for cluster in range(num_clusters):
    cluster_sum = 0  
    cluster_sum = pairwise_distances(enc_output[members == cluster]).mean()  
    sum += cluster_sum
  sum = sum/num_clusters
  avg_dist.append(sum)

print(avg_dist)

In [ ]:
distance_txt = np.savetxt("/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/avg_dist0.txt", avg_dist)

In [ ]:
avg_dist = np.loadtxt('/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/avg_dist0.txt')
# avg_dist = avg_dist.astype(int) # for autoencoder only

In [ ]:
plt.rcParams['lines.linewidth'] = 10

fig = plt.figure(figsize=(30, 18))

fig.patch.set_facecolor('slateblue')

plt.xlabel("Number of Phenotypes", fontsize=50, color='white')
plt.ylabel("Intra-cluster Distance", fontsize=50, color='white')

plt.plot(range(1, 50), avg_dist, color='mediumvioletred')

# Show every 2nd value
plt.xticks(range(1, 50, 2))

ax = plt.gca()
ax.set_facecolor('midnightblue')

ax.tick_params(axis='x', labelsize=25, colors='white')
ax.tick_params(axis='y', labelsize=25, colors='white')

plt.tight_layout()

plt.savefig(
    '/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/elbow0.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

reduction = np.diff(avg_dist)

plt.figure(figsize=(30, 18))

plt.plot(
    range(2, 50),
    reduction,
    linewidth=10,
    color='mediumvioletred'
)

plt.xlabel("Number of Phenotypes", fontsize=50, color='white')
plt.ylabel("Reduction in Intra-cluster Distance", fontsize=50, color='white')

plt.xticks(range(2, 50, 2))

ax = plt.gca()
ax.set_facecolor('midnightblue')
ax.figure.patch.set_facecolor('slateblue')

ax.tick_params(axis='x', labelsize=25, colors='white')
ax.tick_params(axis='y', labelsize=25, colors='white')

plt.tight_layout()

plt.savefig(
    '/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/reduction.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
test1 = []
for i in range(1,19):    
    test = (avg_dist[i-1] - avg_dist[i]) / avg_dist[i-1]
    test1.append(test)

print(test1)

# top 3 reductions, largest first
order = sorted(range(len(test1)), key=lambda j: test1[j], reverse=True)
top3 = order[:3]
for rank, j in enumerate(top3, 1):
    k = j + 2                      # test1[j] is the drop into k = j+2 clusters
    print(f"#{rank}: k={k}  reduction={test1[j]:.4f}")

# final clustering — they hardcoded n_clusters=2; change this to the k printed by Cell 28 if it differs)

In [ ]:
cluster = AgglomerativeClustering(n_clusters=6, metric='euclidean', linkage='ward')
members = cluster.fit_predict(enc_output)

# label assignment

In [ ]:
labels = members

# save labels

In [ ]:
labelstxt = np.savetxt("/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/labels.txt", labels)

# UMAP

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap

# enc_output = (N subjects, latent dimensions)
# labels = cluster assignment for each subject

print("Embedding shape:", enc_output.shape)
print("Labels shape:", labels.shape)
print("Clusters:", np.unique(labels))

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

min_dist        = 0.0125
output_dir      = ("/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k")
neighbor_values = range(5, 400, 10) # n_neighbors: 5, 15, 25, ..., 395

# ------------------------------------------------------------
# Generate UMAP projections
# ------------------------------------------------------------

unique_labels = np.unique(labels)

for n_neighbors in neighbor_values:

    print(f"Running UMAP with n_neighbors = {n_neighbors}...")

    # --------------------------------------------------------
    # UMAP
    # --------------------------------------------------------
    umap_reducer = umap.UMAP( n_neighbors =n_neighbors,
                              min_dist    =min_dist,
                              n_components=2,
                              metric      ="euclidean",
                              random_state=42)

    umap_embedding = umap_reducer.fit_transform(enc_output) # we apply fit transform because we want to reduce the dimensionality of our data to 2D for visualization

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------
    plt.figure(figsize=(10, 8))

    scatter = plt.scatter(
        umap_embedding[:, 0],
        umap_embedding[:, 1],
        c=labels,
        cmap="tab10",
        s=35,
        alpha=0.85
    )

    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title(
        f"UMAP projection of airway latent representations\n"
        f"n_neighbors = {n_neighbors}"
    )

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------
    handles, _ = scatter.legend_elements()

    plt.legend(
        handles,
        [f"Cluster {c}" for c in unique_labels],
        title="Phenotype"
    )

    plt.tight_layout()

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------
    save_path = (
        f"{output_dir}/UMAP_clusters_neighbors_{n_neighbors}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Saved: {save_path}")

print("All UMAP projections generated.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# enc_output = (N subjects, latent dimensions)
# labels = cluster assignment for each subject

print("Embedding shape:", enc_output.shape)
print("Labels shape:", labels.shape)
print("Clusters:", np.unique(labels))

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

output_dir        = ("/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k")
perplexity_values = range(10, 401, 10) # Perplexity: 10, 20, 30, ..., 400
unique_labels     = np.unique(labels)

# ------------------------------------------------------------
# Generate t-SNE projections
# ------------------------------------------------------------

for perplexity in perplexity_values:

    print(f"Running t-SNE with perplexity = {perplexity}...")

    # --------------------------------------------------------
    # t-SNE
    # --------------------------------------------------------
    tsne = TSNE( n_components=2,
                 perplexity=perplexity,
                 init="pca", 
                 random_state=42,
                 max_iter=1000)

    tsne_embedding = tsne.fit_transform(enc_output)

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------
    plt.figure(figsize=(10, 8))

    scatter = plt.scatter(tsne_embedding[:, 0],
                          tsne_embedding[:, 1],
                          c=labels,
                          cmap="tab10",
                          s=35,
                          alpha=0.85)

    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.title(f"t-SNE projection of airway latent representations\n"
              f"perplexity = {perplexity}")

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------
    handles, _ = scatter.legend_elements()
    plt.legend(handles, [f"Cluster {c}" for c in unique_labels], title="Phenotype")
    plt.tight_layout()

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------
    save_path = (f"{output_dir}/tSNE_clusters_perplexity_{perplexity}.png")
    plt.savefig(save_path, dpi=300,bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}")

print("All t-SNE projections generated.")

# Colour coded branch numbers and generations 

In [ ]:
import re                                                       # parse the tree-parse log lines

out_path      = "/home/ids/gmargari-24/airway_project/jobs/step_3_tree_parse_965981.out"  # fix path
labels_path   = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/labels.txt"
manifest_path = None                                            # DGCI case list in row order, if one exists

# --- 1. pull branch count and max generation out of the .out file -------------
line_pat = re.compile(r"^(\S+): (\d+) branches, max generation (\d+)")  # one line per case

case_names, branches, max_gen = [], [], []                      # filled in log order
for line in open(out_path):
    match = line_pat.match(line)                                # header/footer lines won't match
    if match is None:
        continue
    case_names.append(match.group(1))
    branches.append(int(match.group(2)))
    max_gen.append(int(match.group(3)))

branches = np.array(branches)                                   # arrays so we can mask by cluster
max_gen  = np.array(max_gen)
print(f"parsed {len(case_names)} cases from the .out file")

# --- 2. load labels and align them to the cases -------------------------------
labels = np.loadtxt(labels_path).astype(int)                    # savetxt wrote them as floats
assert len(labels) == len(case_names), "label count != case count — mismatched files"

if manifest_path:                                               # preferred: align by case id
    row_names  = [l.strip() for l in open(manifest_path) if l.strip()]
    stats      = {n: (b, g) for n, b, g in zip(case_names, branches, max_gen)}
    branches   = np.array([stats[n][0] for n in row_names])     # reorder to embedding row order
    max_gen    = np.array([stats[n][1] for n in row_names])
    case_names = row_names
else:                                                           # fallback: both sides sorted by case id
    idx        = np.argsort(case_names)                         # no-op, the log is already sorted
    case_names = [case_names[i] for i in idx]
    branches   = branches[idx]
    max_gen    = max_gen[idx]

case_names = np.array(case_names)                               # array so boolean masks work on it

# --- 3. per-cluster means -----------------------------------------------------
print(f"\n{'cluster':>7} {'n':>4} {'branches':>18} {'max generation':>18}")
for c in np.unique(labels):
    mask = labels == c                                          # rows in this cluster
    n    = int(mask.sum())
    b_sd = branches[mask].std(ddof=1) if n > 1 else np.nan      # sd undefined for a single member
    g_sd = max_gen[mask].std(ddof=1) if n > 1 else np.nan
    print(f"{c:>7} {n:>4}   {branches[mask].mean():6.1f} ± {b_sd:<7.1f}"
          f"   {max_gen[mask].mean():6.2f} ± {g_sd:<7.2f}")

# --- 4. list members of the small clusters ------------------------------------
for c in np.unique(labels):
    mask = labels == c
    if mask.sum() > 20:                                         # skip printing the bulk cluster
        continue
    print(f"\ncluster {c} members:")
    for name, b, g in zip(case_names[mask], branches[mask], max_gen[mask]):
        print(f"  {name}: {b} branches, max generation {g}")

# --- 5. save the per-subject table ---------------------------------------------
csv_path = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/cluster_assignments.csv"
with open(csv_path, "w") as f:
    f.write("case,cluster,branches,max_generation\n")           # header
    for name, c, b, g in zip(case_names, labels, branches, max_gen):
        f.write(f"{name},{c},{b},{g}\n")
print(f"\nwrote {csv_path}")

# UMAP and t-SNE of the clustered subjects good visualisation
Projects the same `enc_output` matrix that ward clustered into 2D, then colours each subject by its cluster in `labels`. Every cluster keeps the same colour and marker in both panels.

In [ ]:
# --- UMAP and t-SNE of the clustered embeddings, coloured by cluster ----------
import numpy as np
import matplotlib.pyplot as plt
import umap                                                     # umap-learn package
from sklearn.manifold import TSNE

save_dir = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k"

X      = np.asarray(enc_output, dtype=np.float64)              # the exact matrix ward clustered
labels = np.asarray(labels).astype(int)                        # from the clustering cell / labels.txt
assert len(labels) == X.shape[0], "labels and embeddings have different lengths"

# --- 1. project to 2D (same settings as Registration/umap.py and visualize_embeddings.py)
umap_2d = umap.UMAP(n_neighbors=20, min_dist=0.0125, n_components=2,metric="euclidean", random_state=42).fit_transform(X)

# perp    = min(20.0, (X.shape[0] - 1) / 3.0)                    # t-SNE needs perplexity < N/3
tsne_2d = TSNE(n_components=2, perplexity=50, init="pca", random_state=0, max_iter=1000).fit_transform(X)

# --- 2. one fixed colour + marker per cluster, identical in both panels ---------
palette  = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
markers  = ["o", "s", "^", "D", "P", "X", "v", "*"]             # shape backs up colour (colour-blind safe)
clusters = np.unique(labels)
assert len(clusters) <= len(palette), "more clusters than colours: extend palette and markers"
style = {c: (palette[i], markers[i]) for i, c in enumerate(clusters)}
order = sorted(clusters, key=lambda c: -(labels == c).sum())   # draw big clusters first, small ones on top

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, coords, name in [(axes[0], umap_2d, "UMAP"), (axes[1], tsne_2d, "t-SNE")]:
    for c in order:
        m = labels == c
        colour, marker = style[c]
        ax.scatter(coords[m, 0], coords[m, 1], c=colour, marker=marker,
                   s=40 if m.sum() > 20 else 90,               # enlarge small clusters so they stay visible
                   alpha=0.85, edgecolors="white", linewidths=0.6,
                   label=f"cluster {c} (n={m.sum()})")
    ax.set_title(f"{name} of latent codes: {len(clusters)} ward clusters", fontsize=14)
    ax.set_xlabel(f"{name} 1", fontsize=12)
    ax.set_ylabel(f"{name} 2", fontsize=12)
    ax.grid(color="#e5e5e5", linewidth=0.6)
    ax.set_axisbelow(True)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)

handles, texts = axes[0].get_legend_handles_labels()
pairs = sorted(zip(texts, handles), key=lambda t: int(t[0].split()[1]))   # legend in cluster-id order
fig.legend([h for _, h in pairs], [t for t, _ in pairs], loc="center right", frameon=False, fontsize=11, markerscale=1.3)
fig.tight_layout(rect=[0, 0, 0.88, 1])
fig.savefig(f"{save_dir}/umap_tsne_clusters.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# --- interactive version: hover a point to see which subject it is -------------
import pandas as pd
import plotly.express as px

have_names = "case_names" in globals() and len(case_names) == len(labels)   # built in the tree-parse cell
names = list(case_names) if have_names else [f"row_{i}" for i in range(len(labels))]

for coords, name in [(umap_2d, "UMAP"), (tsne_2d, "t-SNE")]:
    df = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1],
                       "cluster": [f"cluster {c}" for c in labels], "case": names})
    fig = px.scatter(df, x="x", y="y", color="cluster", symbol="cluster", hover_name="case",
                     color_discrete_map={f"cluster {c}": style[c][0] for c in clusters},
                     category_orders={"cluster": [f"cluster {c}" for c in clusters]},
                     labels={"x": f"{name} 1", "y": f"{name} 2"},
                     title=f"{name} of latent codes (hover for subject id)")
    fig.update_traces(marker=dict(size=8, opacity=0.85, line=dict(width=0.5, color="white")))
    fig.update_layout(width=850, height=650, plot_bgcolor="white")
    fig.show()


# Morphological metrics projected onto the latents

Averages the 6 morphological metrics over the branches of each tree, aligns them to the
latent-code rows via `latent_codes.csv`, and colours the UMAP / t-SNE projections by each
metric in turn. Needs `morphology_projection.py` next to this notebook.


In [ ]:
# --- subjects in latent-code row order, then the mean metric per tree ---------
import numpy as np
import pandas as pd
import morphology_projection as mp                              # the helper module, same folder

latent_csv = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/latent_codes.csv"
cases_root = "/home/ids/gmargari-24/airway_project/airmorph_cases"
save_dir   = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k/morphology"

mp.METRIC_NAMES = [                                             # rename to your real metric names
    "divergence", "length", "complexity", "tortuosity", "stenosis", "ectasia",
]

latent_df = pd.read_csv(latent_csv)                             # same row order as latent_codes.npy
subjects  = latent_df["subject"].tolist()                       # 'AIIB23_30_R_mesh.mat', ...
assert len(subjects) == enc_output.shape[0], "csv rows != embedding rows — wrong file?"

metrics = mp.build_metric_table(subjects, cases_root, mp.METRIC_NAMES)   # (N, 6) means, row-aligned
metrics.to_csv(f"{save_dir}/mean_morphological_features.csv", index=False)   # keep for later stats
metrics.head()


In [ ]:
# --- 6 UMAP + 6 t-SNE panels, one per metric ----------------------------------
import umap                                                     # umap-learn package
from sklearn.manifold import TSNE

X = np.asarray(enc_output, dtype=np.float64)                    # the exact matrix ward clustered

if "umap_2d" not in globals():                                  # reuse the projection if already computed
    umap_2d = umap.UMAP(n_neighbors=20, min_dist=0.0125, n_components=2,
                        metric="euclidean", random_state=42).fit_transform(X)
if "tsne_2d" not in globals():
    tsne_2d = TSNE(n_components=2, perplexity=50, init="pca",
                   random_state=0, max_iter=1000).fit_transform(X)

table, figures = mp.project_metrics(umap_2d, tsne_2d, subjects, save_dir,
                                    root=cases_root, metric_names=mp.METRIC_NAMES)
for path in figures:                                            # 12 panels + 2 overview grids
    print(path)


In [ ]:
# --- optional: per-cluster metric means, to name the phenotypes ----------------
summary = metrics[mp.METRIC_NAMES].copy()                       # drop the subject id column
summary["cluster"] = np.asarray(labels).astype(int)             # from the ward clustering cell

means = summary.groupby("cluster").mean()                       # average metric per phenotype
counts = summary.groupby("cluster").size().rename("n")          # cluster sizes for context
print(pd.concat([counts, means], axis=1).round(3))


## Whole-tree counts on the same projections

Two extra panels per projection — branch count and deepest generation — read from
`tree_counts.csv` (written by `count_branches_generations.py`). Run this after the
cell that builds `umap_2d` / `tsne_2d`.


In [ ]:
# --- 2 UMAP + 2 t-SNE panels: branches and deepest generation -----------------
import morphology_projection as mp                              # restart the kernel to pick up edits

tree_counts_csv = "/home/ids/gmargari-24/airway_project/airmorph_cases/tree_counts.csv"

counts, count_figures = mp.project_tree_counts(umap_2d, tsne_2d, subjects,
                                               tree_counts_csv, save_dir)
for path in count_figures:                                      # 4 panels
    print(path)

# counts.describe()                                               # sanity-check the spread


In [ ]:
# --- do the latents actually encode size and depth? ---------------------------
from scipy.stats import spearmanr

# take the column names from the tables themselves, not from mp.METRIC_NAMES —
# a reload resets that constant back to the placeholders in the file.
metric_columns = [c for c in metrics.columns if c != "subject"]  # whatever the metrics table was built with
count_columns = [c for c in counts.columns if c != "subject"]    # n_branches, max_generation

merged = metrics.merge(counts, on="subject", how="left")         # how='left' preserves latent row order
assert len(merged) == len(metrics), "merge changed the row count — duplicate subject ids?"
merged["cluster"] = np.asarray(labels).astype(int)               # safe: row order is unchanged

for column in count_columns:                                     # rank correlation with each metric
    print(f"\n{column} vs:")
    for metric in metric_columns:
        pair = merged[[column, metric]].dropna()                 # Spearman can't handle NaN
        rho, p = spearmanr(pair[column], pair[metric])
        flag = "*" if p < 0.05 else " "                          # crude significance marker
        print(f"  {metric:<28} rho={rho:+.3f}  p={p:.3g} {flag}  n={len(pair)}")

print("\nper-cluster means:")                                    # are clusters split by tree size?
print(merged.groupby("cluster")[count_columns].agg(["mean", "std", "size"]).round(2))
